# Convolutional Neural Networks on MNIST

This notebook builds a small convolutional neural network (CNN) to classify handwritten digits from
the MNIST dataset, then shows how **data augmentation** changes what the network learns. It is the
second CNN demo of Unit 2: the first showed that fully-connected networks cannot use the spatial
structure of an image at all — here we build the architecture that can, and look inside it.

## Learning objectives

- Build a small CNN in Keras for image classification.
- Train and evaluate any Keras model with one reusable helper, `helpers.train_and_evaluate`.
- Inspect the feature maps produced by the early convolutional layers.
- Compare training on raw images versus randomly augmented images.

## Background

You should be comfortable building and compiling a `Sequential` Keras model, and reading a
confusion matrix and classification report. You should also already have the argument from
`U2-2_CNN-1_DenseFails`: a `Dense` layer treats every pixel as an independent input, so flattening
an image throws away adjacency that the layer could not have used anyway.

One detail about the data matters before the first code cell. Keras convolutional layers expect a
**channel axis**, so MNIST's `(N, 28, 28)` arrays are reshaped to `(N, 28, 28, 1)` — the trailing
1 is the single gray-scale channel. Color images would carry a 3 there instead.

## This notebook covers

1. Building, training, and evaluating a CNN on the raw digits, then visualizing its feature maps
2. Repeating the process with on-the-fly data augmentation
3. Review

**Prerequisites:** `U2-1_Images-1_SkimageCV2.ipynb` for convolution with a fixed kernel;
`U2-2_CNN-1_DenseFails.ipynb` for why a convolutional architecture is needed at all.

**Dataset:** MNIST handwritten digits, loaded via `tensorflow.keras.datasets.mnist`.

**References:** https://keras.io/api/layers/convolution_layers/convolution2d/

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',100)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

# Shared course helpers (msds565_helpers.py lives in the repo root).
# Notebooks sit two folders below the root, so '../..' points back to it.
import sys
sys.path.append('../..')
import msds565_helpers as helpers

## 1. Un-augmented MNIST

We first train a small CNN on the raw MNIST digits — no data augmentation.

### 1.1 Load and reshape the data

In [ ]:
from tensorflow.keras.datasets import mnist

# Load data
(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_train = X_train.reshape(-1, 28, 28, 1)
X_test  = X_test.reshape(-1, 28, 28, 1)

# Print shapes
print("X_train.shape:", X_train.shape)
print("X_test.shape: ", X_test.shape)

### 1.2 Build the model

A 2D convolution slides a small **learnable** kernel $K$ over the input image $I$, computing at each
location a weighted sum of the surrounding pixels:

$$ (I * K)[i, j] = \sum_{m}\sum_{n} I[i+m,\, j+n]\; K[m, n] $$

This is the identical operation `U2-1_Images-1_SkimageCV2` applied with hand-designed Sobel and
blur kernels. The only difference — and it is the whole difference — is that here the entries of
$K$ are weights learned by gradient descent rather than numbers someone wrote down.

Three properties follow, and they are exactly what a dense layer lacked:

- **Locality.** Each kernel is 3×3, so it can only combine neighboring pixels. Adjacency is built
  into the architecture rather than something the network must discover.
- **Weight sharing.** The *same* kernel is applied at every position, so a feature learned in one
  corner is detected everywhere. This is also why a `Conv2D(16, (3,3))` layer has only a few
  hundred parameters where a `Dense` layer would need hundreds of thousands.
- **Tolerance to small shifts.** `MaxPooling2D` halves each spatial dimension by keeping only the
  strongest response in each 2×2 window, so a feature that moves a pixel or two produces the same
  output.

Stacking such layers lets each one detect increasingly abstract patterns — edges → textures → digit
parts. The final softmax turns the network's outputs into class probabilities $\hat{p}_c$, and we
train against the true label $y$ with the (sparse) categorical cross-entropy loss:

$$ \mathcal{L} = -\sum_{c} y_c \, \log \hat{p}_c $$

`sparse_categorical_crossentropy` is the same loss as `categorical_crossentropy`, except it takes
integer labels (`3`) instead of one-hot vectors (`[0,0,0,1,0,...]`) — which is what MNIST gives us.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

dropout_rate = 0.1

n_classes  = np.unique(y_train).shape[0]

# Create model
model = Sequential([
    Input(shape=X_train.shape[1:]),
    
    Conv2D(16, (3, 3), padding='same'),
    #BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(),
    
    Conv2D(32, (3, 3), padding='same'),
    #BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(),
    
    #GlobalAveragePooling2D(),
    Flatten(),
    
    Dense(32, activation='relu'),
    Dropout(dropout_rate),
    Dense(16, activation='relu'),
    
    Dense(n_classes, activation='softmax'),
])

# Define the optimizer with a custom learning rate
optimizer = Adam(learning_rate=0.01)

# Compile model
model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitor validation loss
    patience=10,          # Stop after 5 epochs without improvement
    restore_best_weights=True  # Restore the best weights after stopping
)

# Display model summary
model.summary()

### 1.3 Train and evaluate

In [ ]:
# train_and_evaluate (from msds565_helpers) fits the model, then plots the loss
# curves next to the test-set confusion matrix and prints the classification
# report. It returns the fitted model plus its training history.
model, history = helpers.train_and_evaluate(
    model, X_train, y_train, X_test, y_test,
    epochs=10, batch_size=512,
    callbacks=[early_stopping],
)

### 1.4 Visualize convolutional channels

Each filter in the first `Conv2D` layer produces one **feature map** — the response of that filter
at every position in the image. Plotting them shows what the layer actually learned to look for.

`helpers.visualize_layer_outputs(model, image, n)` builds a truncated model ending at layer `n` and
plots every output channel. We stop at the *activation* rather than the raw convolution, so the
ReLU has already zeroed the negative responses and each map shows only where its filter fired.

Expect edge and stroke detectors: bright bands along one orientation of the digit, dark elsewhere.
Compare them to the hand-designed Sobel kernels from `U2-1_Images-1_SkimageCV2` — nobody specified
these, they fell out of gradient descent.

In [ ]:
idx = np.random.choice( range(X_train.shape[0]), 1 )[0]

# Choose an image from your dataset
sample_img = X_train[idx]  # or any image shaped like your input

In [ ]:
# Visualize the first conv block's activation output (Conv2D=0, Activation=1)
helpers.visualize_layer_outputs(model, sample_img, n=1)

## 2. Augmented MNIST

Now we retrain — with the same evaluation helper — but feed the model **augmented** images so it
must learn features that survive those transformations.

**Data augmentation** applies random, label-preserving transformations (rotations, shifts, zooms)
to each training image. Because the transformation is redrawn every epoch, the network effectively
never sees the exact same image twice: it cannot memorize individual training pictures, which
combats overfitting and pushes it toward features that are robust to the transformations.

Worth keeping in mind from `U2-2_CNN-1_DenseFails`: augmentation was the *only* way to make a dense
network tolerate shifted digits. A CNN already has some of that tolerance built in through weight
sharing and pooling, so here augmentation is an enhancement rather than a rescue — which is why the
augmented model below can be *smaller* than the one in section 1 and still hold up.

### 2.1 Reload the data

Section 1 has already consumed `X_train`, so we reload from scratch to keep the two runs
independent.

In [ ]:
from tensorflow.keras.datasets import mnist

# Load data
(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_train = X_train.reshape(-1, 28, 28, 1)
X_test  = X_test.reshape(-1, 28, 28, 1)

# Print shapes
print("X_train.shape:", X_train.shape)
print("X_test.shape: ", X_test.shape)

### 2.2 Build the model

Two deliberate changes from section 1's architecture, both worth noticing:

- **Batch normalization is switched on** after each convolution. Augmented batches vary more from
  step to step, and normalizing each layer's activations keeps training stable.
- **`GlobalAveragePooling2D` replaces `Flatten`.** Flattening preserves *where* each feature was
  found and hands all of it to the dense layers; global average pooling collapses each feature map
  to a single number — its average response over the whole image — keeping only *whether* the
  feature was present. That discards position deliberately, which is what we want when the digit
  may have been shifted anywhere in the frame. It also shrinks the model considerably.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

dropout_rate = 0.2

n_classes  = np.unique(y_train).shape[0]

# Create model
model = Sequential([
    Input(shape=X_train.shape[1:]),
    
    Conv2D(8, (3, 3), padding='same'),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(),
    
    Conv2D(16, (3, 3), padding='same'),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(),
    
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    
    GlobalAveragePooling2D(),
    #Flatten(),
    
    Dense(32, activation='relu'),
    Dropout(dropout_rate),
    Dense(16, activation='relu'),
    
    Dense(n_classes, activation='softmax'),
])

# Define the optimizer with a custom learning rate
optimizer = Adam(learning_rate=0.01)

# Compile model
model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitor validation loss
    patience=25,          # Stop after 5 epochs without improvement
    restore_best_weights=True  # Restore the best weights after stopping
)

# Display model summary
model.summary()

### 2.3 Augment the data, then train and evaluate

`ImageDataGenerator` produces randomly transformed batches on the fly. We pass that generator to
the same `helpers.train_and_evaluate` call in place of the raw training arrays.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define the image generator with augmentation options
datagen = ImageDataGenerator(
    rotation_range=30,      # Rotate images randomly
    width_shift_range=0.2,  # Randomly shift the width of images
    height_shift_range=0.2, # Randomly shift the height of images
    zoom_range=0.2,         # Randomly zoom
)

In [ ]:
# For the augmented run we hand the model a data generator instead of raw arrays:
# pass datagen.flow(...) as X_train and set y_train=None (the generator supplies
# the labels). We still pass the raw X_test / y_test so the confusion matrix and
# classification report are computed on the un-augmented test set.
model, history = helpers.train_and_evaluate(
    model,
    datagen.flow(X_train, y_train, batch_size=512, shuffle=True),  # augmented training batches
    None,                                                          # labels come from the generator
    X_test, y_test,                                                # raw test data for evaluation
    epochs=5,
    callbacks=[early_stopping],
    val_data=datagen.flow(X_test, y_test),                         # validate on augmented batches too
)

### 2.4 Visualize convolutional channels

The same view as section 1.4, on the augmented model. Note the layer index is `n=2` rather than
`n=1` here: this architecture inserts a `BatchNormalization` between the convolution and its
activation, so the activation sits one layer deeper.

Compare these maps against section 1.4's. Filters trained on augmented data tend to look less
tuned to one specific stroke position and more like general-purpose edge detectors — the network
could not rely on the digit sitting in the same place every time.

In [ ]:
idx = np.random.choice( range(X_train.shape[0]), 1 )[0]

# Choose an image from your dataset
sample_img = X_train[idx]  # or any image shaped like your input

In [ ]:
# Visualize the first conv block's activation output (Conv2D=0, BatchNormalization=1, Activation=2)
helpers.visualize_layer_outputs(model, sample_img, n=2)

## 3. Review

| | Section 1 — raw digits | Section 2 — augmented digits |
|---|---|---|
| First conv block | `Conv2D(16)` → ReLU → pool | `Conv2D(8)` → BatchNorm → ReLU → pool |
| Head | `Flatten` → Dense | `GlobalAveragePooling2D` → Dense |
| Training data | Fixed arrays | `datagen.flow(...)` batches, redrawn each epoch |
| Evaluated on | Raw test set | Raw test set (identical, so the numbers compare) |

**Takeaways**

- **A `Conv2D` layer is the notebook-1 argument turned into code.** Locality, weight sharing, and
  pooling are the three properties a `Dense` layer lacked, and all three are structural — the
  network gets them before it has seen a single training image.
- **Convolution is cheap in parameters.** A 3×3 kernel has nine weights no matter how large the
  image, because the same kernel is reused at every position. That is why the CNN in
  `U2-2_CNN-1_DenseFails` beat a million-parameter dense network with a fraction of the weights.
- **Feature maps are readable.** The first block learns edge and stroke detectors that resemble the
  hand-designed Sobel kernels from `U2-1_Images-1_SkimageCV2` — arrived at by gradient descent
  rather than by design. Later layers combine them into parts, and it is worth re-running
  `visualize_layer_outputs` at a deeper `n` to watch that happen.
- **`Flatten` and `GlobalAveragePooling2D` encode different assumptions.** Flatten keeps *where* a
  feature was; global average pooling keeps only *whether* it was there. When the object can appear
  anywhere in the frame, throwing away position is a feature, not a loss.
- **Augmentation helps a CNN, but does not rescue it.** For the dense network in the previous
  notebook, augmentation was the only route to shift-tolerance. Here the architecture already
  supplies some, so augmentation mostly buys robustness and a defense against overfitting — which
  is why the smaller section-2 model holds up.

**Next:** `U2-2_CNN-3_Imbalanced.ipynb` keeps this architecture but breaks the class balance, and
asks what accuracy hides when some digits are far rarer than others.